# OYKHCHAR LoRA Training - SDXL (FREE)

**Why SDXL instead of FLUX:**
- SDXL fits in 16GB GPU (Kaggle free tier)
- Excellent character consistency
- Proven, reliable training
- Still produces high-quality results

**Training Time:** 30-45 minutes on T4 GPU

In [ ]:
# Install dependencies
!pip install -q diffusers[torch] transformers accelerate peft safetensors bitsandbytes
print("[OK] Dependencies installed")

In [ ]:
# Setup Hugging Face authentication
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ['HF_TOKEN'] = hf_token
    print("[OK] Hugging Face token loaded from secrets")
except:
    print("[ERROR] HF_TOKEN not found in secrets!")
    raise Exception("HF_TOKEN required")

!huggingface-cli login --token $HF_TOKEN
print("[OK] Logged into Hugging Face")

In [ ]:
# Extract training dataset
from pathlib import Path
import os
import shutil

dataset_path = '/kaggle/input/oykhchar-lora-images'
extract_path = '/kaggle/working/training_data/images'

print(f"Using dataset from {dataset_path}...")
os.makedirs(extract_path, exist_ok=True)

# Copy all images and captions
image_files = list(Path(dataset_path).glob('*.jpg')) + list(Path(dataset_path).glob('*.png'))
caption_files = list(Path(dataset_path).glob('*.txt'))

print(f"Copying {len(image_files)} images...")
for img_file in image_files:
    shutil.copy(img_file, extract_path)
    
for cap_file in caption_files:
    shutil.copy(cap_file, extract_path)

final_images = list(Path(extract_path).glob('*.jpg')) + list(Path(extract_path).glob('*.png'))
final_captions = list(Path(extract_path).glob('*.txt'))

print(f"[OK] Copied {len(final_images)} images")
print(f"[OK] Found {len(final_captions)} captions")
print(f"Dataset location: {extract_path}")

# Show samples
for img in sorted(final_images)[:3]:
    print(f"  {img.name}")
    cap = img.with_suffix('.txt')
    if cap.exists():
        print(f"    -> {cap.read_text().strip()[:60]}...")

In [ ]:
# Check GPU
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    torch.cuda.empty_cache()
    print("[OK] GPU ready")
else:
    print("[ERROR] No GPU! Enable GPU in notebook settings.")

In [ ]:
# Prepare dataset
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

class LoRADataset(Dataset):
    def __init__(self, image_dir, size=1024):
        self.image_paths = sorted(list(Path(image_dir).glob('*.jpg')) + list(Path(image_dir).glob('*.png')))
        self.size = size
        
        self.transform = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        
        # Load captions
        self.captions = []
        for img_path in self.image_paths:
            caption_path = img_path.with_suffix('.txt')
            if caption_path.exists():
                self.captions.append(caption_path.read_text().strip())
            else:
                self.captions.append("OYKHCHAR character")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        image = self.transform(image)
        caption = self.captions[idx]
        return {'pixel_values': image, 'caption': caption}

dataset = LoRADataset(extract_path)
print(f"[OK] Dataset prepared: {len(dataset)} images")

In [ ]:
# Load SDXL base model
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
import torch

print("Loading SDXL model (this takes 5-10 minutes)...")

# Load VAE
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

# Load pipeline
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)

pipe.to("cuda")
print("[OK] SDXL model loaded")

In [ ]:
# Configure LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    lora_dropout=0.1,
)

# Apply LoRA to UNet
pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.print_trainable_parameters()

print("[OK] LoRA configured")

In [ ]:
# Training loop
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import torch.nn.functional as F

# Settings
num_epochs = 50
learning_rate = 1e-4
batch_size = 1

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
optimizer = AdamW(pipe.unet.parameters(), lr=learning_rate)

print("[TRAINING] Starting LoRA training...")
print("="*60)
print(f"Epochs: {num_epochs}")
print(f"Steps per epoch: {len(dataloader)}")
print(f"Total steps: {num_epochs * len(dataloader)}")
print("="*60)
print()

pipe.unet.train()
global_step = 0

for epoch in range(num_epochs):
    epoch_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch in progress_bar:
        pixel_values = batch['pixel_values'].to("cuda", dtype=torch.float16)
        captions = batch['caption']
        
        # Encode images to latents
        latents = pipe.vae.encode(pixel_values).latent_dist.sample()
        latents = latents * pipe.vae.config.scaling_factor
        
        # Sample noise
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        
        # Sample timesteps
        timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=latents.device)
        timesteps = timesteps.long()
        
        # Add noise to latents
        noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
        
        # Encode text
        text_inputs = pipe.tokenizer(
            captions,
            padding="max_length",
            max_length=pipe.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt"
        )
        
        text_embeddings = pipe.text_encoder(
            text_inputs.input_ids.to("cuda"),
            output_hidden_states=True
        )
        
        encoder_hidden_states = text_embeddings.hidden_states[-2]
        
        # Predict noise
        model_pred = pipe.unet(
            noisy_latents,
            timesteps,
            encoder_hidden_states,
            added_cond_kwargs={"text_embeds": text_embeddings[0], "time_ids": torch.zeros((bsz, 6), device="cuda")}
        ).sample
        
        # Calculate loss
        loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        global_step += 1
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1} completed - Avg Loss: {avg_loss:.4f}")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"/kaggle/working/checkpoint_epoch_{epoch+1}"
        pipe.unet.save_pretrained(checkpoint_path)
        print(f"[SAVE] Checkpoint saved: {checkpoint_path}")

print()
print("[OK] Training complete!")

In [ ]:
# Save final LoRA
output_dir = "/kaggle/working/oykhchar_sdxl_lora"
pipe.unet.save_pretrained(output_dir)
print(f"[OK] LoRA weights saved: {output_dir}")
print()
!ls -lh {output_dir}

In [ ]:
# Test generation
print("[TEST] Generating test images...")
pipe.unet.eval()

test_prompts = [
    "OYKHCHAR character standing with arms raised in celebration",
    "OYKHCHAR character sitting and thinking",
    "OYKHCHAR character pointing forward energetically"
]

from IPython.display import display

for i, prompt in enumerate(test_prompts):
    print(f"\nGenerating: {prompt}")
    image = pipe(
        prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        height=1024,
        width=1024
    ).images[0]
    
    image.save(f"/kaggle/working/test_{i+1}.png")
    display(image.resize((512, 512)))
    print(f"[OK] Saved: test_{i+1}.png")

print("\n[DONE] All done! Download your LoRA from the Output tab.")

## Training Complete!

### Your trained LoRA is ready:
- Location: `/kaggle/working/oykhchar_sdxl_lora`
- Download from Output tab

### Using Your LoRA:

```python
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16
)
pipe.load_lora_weights("./oykhchar_sdxl_lora")
pipe.to("cuda")

image = pipe(
    "OYKHCHAR character waving hello",
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]
```

**Remember:** Always include `OYKHCHAR` in your prompts!